In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================
# 1. Load data
# =========================

df = pd.read_csv("skyscanner_airfare_data.csv")
df["WeekNumber"] = df["FlightWeek"].str.extract(r"W(\d+)").astype(int)
df["Route"] = df["OriginAirport"] + "_" + df["DestinationAirport"]
df["CountryRoute"] = df["OriginCountry"] + "_" + df["DestinationCountry"]
df["WeekSin"] = np.sin(2 * np.pi * df["WeekNumber"] / 52)
df["WeekCos"] = np.cos(2 * np.pi * df["WeekNumber"] / 52)
print(df.shape)
print(df.head())
print(df.columns)


(55000, 19)
  FlightWeek OriginAirport OriginCountry DestinationAirport  \
0   2024-W01           AMS            NL                AGP   
1   2024-W01           AMS            NL                AYT   
2   2024-W01           AMS            NL                DBV   
3   2024-W01           AMS            NL                DLM   
4   2024-W01           AMS            NL                DLM   

  DestinationCountry MainAirlineCarrier  IsConnectingFlight UserCountry  \
0                 ES                 LH                   0          CZ   
1                 TR                 FR                   0          BE   
2                 HR                 LH                   0          DE   
3                 TR                 LH                   0          DE   
4                 TR                 EW                   0          AT   

     TripType CabinClass  NumberOfNights  BookingHorizon  NumberOfRedirects  \
0     one-way    Economy              14              68                  4   


In [2]:


# =========================
# 2. Basic cleanup
# =========================

df = df.copy()

# Normalize carrier just in case
df["MainAirlineCarrier"] = df["MainAirlineCarrier"].astype(str).str.upper()

# Drop missing target rows
df = df.dropna(subset=["Revenue", "NumberOfRedirects"])

# Features we can know before prediction
base_features = [
    "WeekNumber",
    "WeekSin",
    "WeekCos",
    "Route",
    "CountryRoute",
    "OriginAirport",
    "OriginCountry",
    "DestinationAirport",
    "DestinationCountry",
    "MainAirlineCarrier",
    "IsConnectingFlight",
    "UserCountry",
    "TripType",
    "CabinClass",
    "NumberOfNights",
    "BookingHorizon",
]

categorical_features = [
    "Route",
    "CountryRoute",
    "OriginAirport",
    "OriginCountry",
    "DestinationAirport",
    "DestinationCountry",
    "MainAirlineCarrier",
    "UserCountry",
    "TripType",
    "CabinClass",
]

numeric_features = [
    "WeekNumber",
    "WeekSin",
    "WeekCos",
    "IsConnectingFlight",
    "NumberOfNights",
    "BookingHorizon",
]


# =========================
# 3. Shared preprocessing
# =========================

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)


In [3]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5
)


def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")


# =========================
# 4. Model 1: competitor price prediction
# Target = Revenue
# Rows = competitors only
# =========================



competitor_df = df[df["MainAirlineCarrier"] != "EW"].copy()
competitor_df = competitor_df.sort_values("FlightWeek")

unique_weeks = sorted(competitor_df["FlightWeek"].unique())

split_point = int(len(unique_weeks) * 0.8)

train_weeks = unique_weeks[:split_point]
test_weeks = unique_weeks[split_point:]
    
train_df = competitor_df[competitor_df["FlightWeek"].isin(train_weeks)]
test_df = competitor_df[competitor_df["FlightWeek"].isin(test_weeks)]

X_train = train_df[base_features]
y_train = train_df["Revenue"]

X_test = test_df[base_features]
y_test = test_df["Revenue"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X_price,
#     y_price,
#     test_size=0.2,
#     random_state=42
# )

price_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model)
    ]
)

price_pipeline.fit(X_train, y_train)

price_preds = price_pipeline.predict(X_test)

evaluate_model(
    "Model 1: Competitor Price Prediction",
    y_test,
    price_preds
)



Model 1: Competitor Price Prediction
----------------------------------------
MAE  : 16.9359
RMSE : 23.9234
R2   : 0.7321


In [4]:
# =========================
# Model 1 Improved: Log Revenue Target
# =========================

# log-transform target
y_train_price_log = np.log1p(y_train)

# train model on log Revenue
price_pipeline.fit(X_train, y_train_price_log)

# predict log Revenue
price_preds_log = price_pipeline.predict(X_test)

# convert predictions back to original Revenue scale
price_preds_log_model = np.expm1(price_preds_log)

# avoid negative values just in case
price_preds_log_model = np.maximum(price_preds_log_model, 0)

evaluate_model(
    "Model 1 Improved: Log Revenue Competitor Price Prediction",
    y_test,
    price_preds_log_model
)

price_mae_log = mean_absolute_error(y_test, price_preds_log_model)

print("\nCompetitor Test Revenue mean:", y_test.mean())
print("Competitor Test Revenue median:", y_test.median())
print("Price MAE as % of test mean:", price_mae_log / y_test.mean() * 100)


Model 1 Improved: Log Revenue Competitor Price Prediction
----------------------------------------
MAE  : 15.8140
RMSE : 22.1260
R2   : 0.7709

Competitor Test Revenue mean: 79.15562542720438
Competitor Test Revenue median: 66.32
Price MAE as % of test mean: 19.97842683350824


In [5]:
pred_check = test_df.copy()
pred_check["ActualRevenue"] = y_test.values
pred_check["PredictedRevenue"] = price_preds_log_model

weekly_pred_check = (
    pred_check
    .groupby("FlightWeek", as_index=False)
    .agg(
        ActualRevenue=("ActualRevenue", "mean"),
        PredictedRevenue=("PredictedRevenue", "mean")
    )
)

weekly_pred_check

,FlightWeek,ActualRevenue,PredictedRevenue
0,2024-W42,92.764126,93.501964
1,2024-W43,90.098839,93.542768
2,2024-W44,85.451519,88.926336
3,2024-W45,86.667285,91.800862
4,2024-W46,83.710324,90.748269
5,2024-W47,76.553974,87.747797
6,2024-W48,74.575883,87.056609
7,2024-W49,72.962732,88.439523
8,2024-W50,69.904198,86.540516
9,2024-W51,68.478284,85.458852


In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except ImportError:
    raise ImportError("Please install statsmodels first: pip install statsmodels")


# =========================
# 1. Helpers
# =========================

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


def safe_expm1(x):
    return np.maximum(0, np.expm1(x))


# =========================
# 2. Load + feature engineering
# =========================

df = pd.read_csv("skyscanner_airfare_data.csv")

df["WeekNumber"] = df["FlightWeek"].str.extract(r"W(\d+)").astype(int)
df["Route"] = df["OriginAirport"] + "_" + df["DestinationAirport"]
df["CountryRoute"] = df["OriginCountry"] + "_" + df["DestinationCountry"]

df["WeekSin"] = np.sin(2 * np.pi * df["WeekNumber"] / 52)
df["WeekCos"] = np.cos(2 * np.pi * df["WeekNumber"] / 52)

df["MainAirlineCarrier"] = df["MainAirlineCarrier"].astype(str).str.upper()

df = df.dropna(subset=["Revenue"]).copy()


# =========================
# 3. Feature lists
# =========================

BASE_FEATURES = [
    "WeekNumber",
    "WeekSin",
    "WeekCos",
    "Route",
    "CountryRoute",
    "OriginAirport",
    "OriginCountry",
    "DestinationAirport",
    "DestinationCountry",
    "MainAirlineCarrier",
    "IsConnectingFlight",
    "UserCountry",
    "TripType",
    "CabinClass",
    "NumberOfNights",
    "BookingHorizon",
]

CATEGORICAL_FEATURES = [
    "Route",
    "CountryRoute",
    "OriginAirport",
    "OriginCountry",
    "DestinationAirport",
    "DestinationCountry",
    "MainAirlineCarrier",
    "UserCountry",
    "TripType",
    "CabinClass",
]

NUMERIC_FEATURES = [
    "WeekNumber",
    "WeekSin",
    "WeekCos",
    "IsConnectingFlight",
    "NumberOfNights",
    "BookingHorizon",
]


# Residual RF should NOT use time features because Holt owns time trend
RESIDUAL_FEATURES = [
    col for col in BASE_FEATURES
    if col not in ["WeekNumber", "WeekSin", "WeekCos"]
]

RESIDUAL_CATEGORICAL_FEATURES = [
    col for col in CATEGORICAL_FEATURES
    if col in RESIDUAL_FEATURES
]

RESIDUAL_NUMERIC_FEATURES = [
    col for col in NUMERIC_FEATURES
    if col in RESIDUAL_FEATURES
]


def build_preprocess(cat_features, num_features):
    return ColumnTransformer(
        transformers=[
            ("cat", make_ohe(), cat_features),
            ("num", "passthrough", num_features),
        ]
    )


# =========================
# 4. Competitor-only time split
# =========================

competitor_df = df[df["MainAirlineCarrier"] != "EW"].copy()
competitor_df = competitor_df.sort_values("WeekNumber")

unique_weeks = sorted(competitor_df["WeekNumber"].unique())

split_point = int(len(unique_weeks) * 0.8)

train_weeks = unique_weeks[:split_point]
test_weeks = unique_weeks[split_point:]

train_df = competitor_df[competitor_df["WeekNumber"].isin(train_weeks)].copy()
test_df = competitor_df[competitor_df["WeekNumber"].isin(test_weeks)].copy()

print("Train weeks:", train_weeks[0], "to", train_weeks[-1])
print("Test weeks :", test_weeks[0], "to", test_weeks[-1])
print("Train rows :", len(train_df))
print("Test rows  :", len(test_df))


# =========================
# 5. Baseline: your current Log RF model
# =========================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5
)

current_preprocess = build_preprocess(
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES
)

current_log_rf_pipeline = Pipeline(
    steps=[
        ("preprocess", current_preprocess),
        ("model", rf_model),
    ]
)

X_train = train_df[BASE_FEATURES]
y_train = train_df["Revenue"]

X_test = test_df[BASE_FEATURES]
y_test = test_df["Revenue"]

current_log_rf_pipeline.fit(X_train, np.log1p(y_train))

current_log_preds = current_log_rf_pipeline.predict(X_test)
current_revenue_preds = safe_expm1(current_log_preds)

metrics = []

metrics.append(
    evaluate_model(
        "Current Model 1: Log RF Revenue",
        y_test,
        current_revenue_preds,
    )
)


# =========================
# 6. Model A: Holt damped trend only
# =========================

train_df["LogRevenue"] = np.log1p(train_df["Revenue"])
test_df["LogRevenue"] = np.log1p(test_df["Revenue"])

weekly_train_log = (
    train_df
    .groupby("WeekNumber")["LogRevenue"]
    .mean()
    .reindex(train_weeks)
)

holt_model = ExponentialSmoothing(
    weekly_train_log,
    trend="add",
    damped_trend=True,
    seasonal=None,
    initialization_method="estimated",
).fit(optimized=True)

trend_train_log = pd.Series(
    holt_model.fittedvalues.values,
    index=train_weeks,
)

trend_test_log = pd.Series(
    holt_model.forecast(len(test_weeks)).values,
    index=test_weeks,
)

train_df["TrendLogPrediction"] = train_df["WeekNumber"].map(trend_train_log)
test_df["TrendLogPrediction"] = test_df["WeekNumber"].map(trend_test_log)

trend_only_preds = safe_expm1(test_df["TrendLogPrediction"].values)

metrics.append(
    evaluate_model(
        "Model A: Holt Damped Trend Only",
        y_test,
        trend_only_preds,
    )
)


# =========================
# 7. Model B: RF learns residual only
# residual = actual log revenue - trend log prediction
# =========================

train_df["ResidualLogRevenue"] = (
    train_df["LogRevenue"] - train_df["TrendLogPrediction"]
)

test_df["ResidualLogRevenue"] = (
    test_df["LogRevenue"] - test_df["TrendLogPrediction"]
)

residual_preprocess = build_preprocess(
    RESIDUAL_CATEGORICAL_FEATURES,
    RESIDUAL_NUMERIC_FEATURES,
)

residual_rf_pipeline = Pipeline(
    steps=[
        ("preprocess", residual_preprocess),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=5,
            ),
        ),
    ]
)

residual_rf_pipeline.fit(
    train_df[RESIDUAL_FEATURES],
    train_df["ResidualLogRevenue"],
)

test_residual_preds = residual_rf_pipeline.predict(
    test_df[RESIDUAL_FEATURES]
)

evaluate_model(
    "Model B: RF Residual Prediction Only - Log Scale",
    test_df["ResidualLogRevenue"],
    test_residual_preds,
)


# =========================
# 8. Final combined model
# final log prediction = Holt trend forecast + RF residual prediction
# =========================

combined_log_preds = (
    test_df["TrendLogPrediction"].values + test_residual_preds
)

combined_revenue_preds = safe_expm1(combined_log_preds)

metrics.append(
    evaluate_model(
        "Final Combined Model: Holt Trend + RF Residual",
        y_test,
        combined_revenue_preds,
    )
)


# =========================
# 9. Final comparison table
# =========================

metrics_df = pd.DataFrame(metrics)

print("\nMODEL COMPARISON")
print("=" * 60)
print(metrics_df)


# =========================
# 10. Weekly actual vs predicted comparison
# =========================

comparison_df = test_df[
    ["FlightWeek", "WeekNumber", "Revenue"]
].copy()

comparison_df["CurrentLogRF_PredictedRevenue"] = current_revenue_preds
comparison_df["HoltTrendOnly_PredictedRevenue"] = trend_only_preds
comparison_df["CombinedTrendResidual_PredictedRevenue"] = combined_revenue_preds

weekly_comparison = (
    comparison_df
    .groupby(["FlightWeek", "WeekNumber"], as_index=False)
    .agg(
        ActualRevenue=("Revenue", "mean"),
        CurrentLogRF=("CurrentLogRF_PredictedRevenue", "mean"),
        HoltTrendOnly=("HoltTrendOnly_PredictedRevenue", "mean"),
        CombinedTrendResidual=("CombinedTrendResidual_PredictedRevenue", "mean"),
    )
    .sort_values("WeekNumber")
)

print("\nWEEKLY COMPARISON")
print("=" * 60)
print(weekly_comparison)


# =========================
# 11. Optional quick plot
# =========================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(weekly_comparison["WeekNumber"], weekly_comparison["ActualRevenue"], marker="o", label="Actual")
plt.plot(weekly_comparison["WeekNumber"], weekly_comparison["CurrentLogRF"], marker="o", label="Current Log RF")
plt.plot(weekly_comparison["WeekNumber"], weekly_comparison["HoltTrendOnly"], marker="o", label="Holt Trend Only")
plt.plot(weekly_comparison["WeekNumber"], weekly_comparison["CombinedTrendResidual"], marker="o", label="Combined Trend + Residual RF")

plt.xlabel("Week Number")
plt.ylabel("Average Revenue")
plt.title("Actual vs Predicted Weekly Revenue")
plt.legend()
plt.grid(True)
plt.show()

Train weeks: 1 to 41
Test weeks : 42 to 52
Train rows : 32638
Test rows  : 8778

Current Model 1: Log RF Revenue
--------------------------------------------------
MAE  : 16.0271
RMSE : 22.1418
R2   : 0.7705

Model A: Holt Damped Trend Only
--------------------------------------------------
MAE  : 27.4424
RMSE : 46.8680
R2   : -0.0281


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
